# 季节性EPS信息来源原型
固定拆股边界四份官方公告，只核对GAAP diluted EPS、同财政季度、拆股及显示精度。原始公告当前副本不等于首版归档；不构造训练信号或评价收益。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from bs4 import BeautifulSoup
from src.data.nvda_seasonal_eps import extract_q2_eps,split_bridge
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/seasonal_eps_source_pilot_20261015';p=json.loads((card/'plan.json').read_text());path=root/p['source_manifest'];assert hashlib.sha256(path.read_bytes()).hexdigest()==p['source_manifest_sha256'];m=json.loads(path.read_text());events={};raw={}
for e in m['entries']:
 data=(path.parent/e['path']).read_bytes();assert e['status']==200 and hashlib.sha256(data).hexdigest()==e['sha256'];year=int(e['year']);raw[str(year)]=data;events[year]={**extract_q2_eps(data.decode(),fiscal_year=year),'source_url':e['url'],'source_sha256':e['sha256']}
body2022=BeautifulSoup(raw['2022'],'html.parser').get_text(' ',strip=True);body2025=BeautifulSoup(raw['2025'],'html.parser').get_text(' ',strip=True)
assert 'four-for-one split' in body2022 and 'July 19, 2021' in body2022;assert 'ten-for-one forward stock split' in body2025 and 'June 7, 2024' in body2025
frame=pd.DataFrame([{'datetime':pd.Timestamp(e['release_date']),'instrument':'NVDA_EPS_EVENT','reported_gaap_diluted_eps':float(e['current_eps']),'reported_prior_year_comparative_eps':float(e['prior_year_comparative_eps']),'training_input_qualified':0.} for e in events.values()]).sort_values('datetime');provider=root/'data/qlib/nvda_seasonal_eps_pilot_20261015_v1'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';manifest=build_qlib_store_from_frame(frame,provider_uri=provider,as_of=m['as_of'],field_map={f:f for f in ['reported_gaap_diluted_eps','reported_prior_year_comparative_eps','training_input_qualified']},source='Dated issuer Q2 GAAP diluted EPS copies; event reference only, no publication clock qualification')
qlib.init(provider_uri=str(provider),region='us');quotes=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['NVDA_EPS_EVENT'],fields=['$reported_gaap_diluted_eps','$reported_prior_year_comparative_eps','$training_input_qualified'],start=str(frame.datetime.min().date()),end=m['as_of']);np.testing.assert_allclose(quotes['$reported_gaap_diluted_eps'],frame.reported_gaap_diluted_eps.to_numpy(),rtol=0,atol=1e-6)
# Lexical decimal precision is retained for the source-rounding audit after native readback.
bridges=pd.DataFrame([split_bridge(events[2021],events[2022],split_ratio=4),split_bridge(events[2024],events[2025],split_ratio=10)]);assert bridges.comparative_rounding_intervals_overlap.all()
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/raw_html','inputs/source_manifest','analysis/quotes']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_manifest=m,raw_html=raw,parser=(root/'src/data/nvda_seasonal_eps.py').read_text());rec.save_objects(artifact_path='analysis',events=events,quotes=quotes,provider_manifest=manifest,split_bridges=bridges);rec.log_metrics(issuer_releases=4,split_boundary_pairs=2,registered_artifact_count=8)
summary={'experiment':card.name,'decision':'needs_full_corpus_and_vintage_clock','new_fits':0,'new_backtests':0,'source_reports':4,'split_pairs':2,'split_rounding_checks_passed':True,'full_sue_series_built':False,'training_input_qualified':False,'consensus_used':False,'provider_uri':str(provider),'region':'us','as_of':m['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
pd.DataFrame(events.values()).to_csv(card/'source_events.csv',index=False);bridges.to_csv(card/'split_bridges.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/'research/ibkr_interest_archive_20261015/nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=p['source_manifest_sha256'],raw_authority='Official issuer dated release current HTML copies, not certified historical first vintage');(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(bridges.to_string(index=False))


# 原始公告季节性EPS输入原型：拆股检查通过，尚不可训练

本轮0拟合、0回测、未查看事件后收益。区别于earnings_enhancement_20261005的供应商EPS/分析师一致预期快照，本方向拟核查同财政季度的季节性盈利预测误差。它不是市场一致预期，也不能称实际超预期或已验证alpha。[相关研究定义](https://www.nber.org/system/files/working_papers/w30860/revisions/w30860.rev1.pdf)使用拆股调整后的同比EPS变化及过去八季度变化的标准差；该论文结果不等于SOXX/SOXL策略有效。

固定核对2021年四拆一和2024年十拆一两处边界，选择各自拆股后Q2报告及上一年同季度报告，共四份官方历史公告的当前HTML副本：FY2021Q2、FY2022Q2、FY2024Q2、FY2025Q2。四份发行人网页均HTTP200并保存原始字节/哈希；本地SEC直接读取样例403未得到原件，不把该错误页视为财务证据。没有绕过登录或修改全局网络策略。

解析限定GAAP diluted EPS，核对财政季度标题、GAAP表与正文数值；不混Non-GAAP或基本EPS，拒绝不相邻的年度同季度配对。原显示小数精度保留，Qlib仅登记事件参考字段，training_input_qualified始终为0，不作向日频交易特征的前向填充。

| 对比 | 上年原报EPS | 当期EPS | 未调拆股的差 | 拆股后同比差 | 当期表内可比差 |
|---|---:|---:|---:|---:|---:|
| FY2021Q2→FY2022Q2，4拆1 | 0.99 | 0.94 | -0.05 | +0.6925 | +0.69 |
| FY2024Q2→FY2025Q2，10拆1 | 2.48 | 0.67 | -1.81 | +0.422 | +0.42 |

拆股倍数和完成日期来自对应当期公告正文；换算后的上年数值与当期表内上年数值的±半显示单位区间相交，说明观察差异与显示舍入相容，不证明不存在任何其他重述。表中换算小数是算术派生值，不能当作发行人报告了更高测量精度。若忽略股本口径，两例均会把增长错写为下降。

来源：[FY2021Q2](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2021)、[FY2022Q2](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2022)、[FY2024Q2](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2024)、[FY2025Q2](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2025/)。

决策needs_full_corpus_and_vintage_clock。当前页面副本不等于历史首版；日期不等于精确公开时刻，旧SEC API/index时钟分歧亦未由此解决。仅四份公告不足以构建连续同比变化或八个既往变化的尺度；若严格排除当前变化，需至少13个连续季度的原值/比较关系、已知拆股处理与口径审计。尚未计算SUE、跨公司排名或ETF合成，也没有宣称单一NVDA代表完整历史SOXX成分。

下一步先做完整季度原件/口径和时点资格，再冻结唯一指标与对照；不能把已有供应商失败路线换名字重跑，不能拿当期重新列示的上年数值冒充此前已知预期。任何收益检验仍须时间隔离和独立证据。生产不变，稳定净SOXX超额未完成。

1项聚焦合同测试通过；实际四公告解析、原生Qlib读回及NAS冷恢复逐对象哈希通过。原始HTML与精确字符串事实在Recorder，瘦卡只保留事件摘要与两组口径检查。
